# Data exploration: grounding the README's data statements

This notebook checks every factual statement in **README section Data preparation** against the
raw dataset.

**Run it** from the repository root (the first run downloads the dataset, about 100 MB):

```bash
uv run --with jupyter jupyter lab data_exploration.ipynb
```

In [1]:
import re

import pandas as pd
from datasets import load_dataset
from huggingface_hub import HfApi

from news_mcp.config import DATASET, DATASET_SPLIT
from news_mcp.ingest import MODELS, clean_article, repair_text

pd.set_option("display.width", 140)
pd.set_option("display.max_colwidth", 120)

raw = load_dataset(DATASET, split=DATASET_SPLIT)
df = raw.to_pandas()
body = df["article body"].fillna("")

In [2]:
try:
    revision = HfApi().dataset_info(DATASET).sha
except Exception as error:  # offline: the cached copy is used
    revision = f"unknown ({type(error).__name__})"
print(f"{DATASET}, split={DATASET_SPLIT}, revision={revision}")
print(f"{len(df):,} rows; columns: {list(df.columns)}")

eoplumbum/v4_nuclear_power_articles, split=train, revision=0586cbf16aaca410eb24f711166ed5b9380f5bf2
26,483 rows; columns: ['id', 'country', 'language', 'date', 'year', 'source', 'url', 'title', 'article body', 'gemini_article_rating', 'gemini_headline_sentiment', 'gemini_headline_alarmistic', 'deepseek_article_rating', 'deepseek_headline_sentiment', 'deepseek_headline_alarmistic', 'gpt_article_rating', 'gpt_headline_sentiment', 'gpt_headline_alarmistic']


## Size, outlets and date range



In [3]:
dates = pd.to_datetime(df["date"], format="%d-%m-%Y")
n_articles, n_outlets = len(df), df["source"].nunique()
first, last = dates.min().date().isoformat(), dates.max().date().isoformat()
print(f"First date: {first}, last date: {last}")
print(f"Articles: {n_articles}, outlets: {n_outlets}")

df.groupby(["country", "language", "source"]).size().rename("articles").to_frame()

First date: 1998-07-11, last date: 2025-01-23
Articles: 26483, outlets: 12


articles
country        language source                    
Czech Republic cz       aktualne.cz           2749
                        iROZHLAS              1891
                        novinky.cz            2155
Hungary        hu       Blikk_hun             1102
                        index.hu              3741
                        origo.hun             3136
Poland         pl       Fakt_pl                432
                        Onet_pl               2350
                        Wirtualna Polska       705
Slovakia       sk       cas.sk                1614
                        pravda.sk             1401
                        sme.sk                5207

> The `train` split of `eoplumbum/v4_nuclear_power_articles` has **26,483 articles from 12 outlets**,
> dated **1998-07-11 to 2025-01-23**.

## The three annotators and their scales


In [4]:
rating_columns = [c for c in df.columns if c.startswith(tuple(MODELS))]

print("Rating columns:", rating_columns)

Rating columns: ['gemini_article_rating', 'gemini_headline_sentiment', 'gemini_headline_alarmistic', 'deepseek_article_rating', 'deepseek_headline_sentiment', 'deepseek_headline_alarmistic', 'gpt_article_rating', 'gpt_headline_sentiment', 'gpt_headline_alarmistic']


> Each article was rated by **three LLMs (Gemini 2.0 Flash, GPT-4o, DeepSeek-V3)**, and the ratings are used as given.

In [5]:
ranges = df[rating_columns].agg(["min", "max"]).T
expected = {c: ((0, 5) if c.endswith("article_rating") else (1, 5)) for c in rating_columns}
ranges_ok = all((ranges.loc[c, "min"], ranges.loc[c, "max"]) == expected[c] for c in rating_columns)
print("article_rating is 0-5 (0 = not about nuclear power); the headline scores are 1-5" if ranges_ok else "contradicted")
ranges

article_rating is 0-5 (0 = not about nuclear power); the headline scores are 1-5


,min,max
gemini_article_rating,0,5
gemini_headline_sentiment,1,5
gemini_headline_alarmistic,1,5
deepseek_article_rating,0,5
deepseek_headline_sentiment,1,5
deepseek_headline_alarmistic,1,5
gpt_article_rating,0,5
gpt_headline_sentiment,1,5
gpt_headline_alarmistic,1,5


**Is the alarmism scale really inverted?** The card defines `headline_alarmistic` as 5 = *not*
alarmist, the opposite of what the name suggests. The data allows an independent check. Calm
headlines tend to be optimistic, so if 5 means "calm", headline sentiment and the alarmistic score
should correlate **positively** within each model. If 5 meant "most alarmist", they would correlate
negatively.

In [6]:
inversion = {m: df[f"{m}_headline_sentiment"].corr(df[f"{m}_headline_alarmistic"]) for m in MODELS}

print("in the raw data 5 means not alarmist (hence headline_alarmism = 6 - mean(raw))" if all(r > 0 for r in inversion.values()) else "contradicted",
      "card defines 5 = not alarmist; correlation(headline_sentiment, headline_alarmistic) per model: "
      + ", ".join(f"{m} {r:.2f}" for m, r in inversion.items()))

in the raw data 5 means not alarmist (hence headline_alarmism = 6 - mean(raw)) card defines 5 = not alarmist; correlation(headline_sentiment, headline_alarmistic) per model: gemini 0.71, deepseek 0.78, gpt 0.83


## Raw formats that need cleaning

> `ingest.clean_article` converts dates to ISO; changes the language code `cz` to `cs`.

In [7]:
date_format_ok = df["date"].str.fullmatch(r"\d{2}-\d{2}-\d{4}").all()
codes = df["language"].value_counts()
print("dates need converting to ISO; the language code `cz` needs changing to `cs`" if date_format_ok and "cz" in codes and "cs" not in codes else "contradicted",
      f"all {len(df):,} dates are DD-MM-YYYY (e.g. {df['date'].iloc[0]!r}); language codes: {codes.to_dict()} "
      "(the ISO 639-1 code for Czech is cs)")

dates need converting to ISO; the language code `cz` needs changing to `cs` all 26,483 dates are DD-MM-YYYY (e.g. '18-09-2011'); language codes: {'sk': 8222, 'hu': 7979, 'cz': 6795, 'pl': 3487} (the ISO 639-1 code for Czech is cs)


## Relevance: articles "not about nuclear power"

> `nuclear_related`: at least 2 of 3 models gave `article_rating > 0`; **7.6% of articles are 0**.

The rule is applied with `clean_article`, exactly as during indexing.

In [9]:
cleaned = pd.DataFrame([clean_article(row) for row in raw])
unrelated_share = 1 - cleaned["nuclear_related"].mean()
zero_votes = (df[[f"{m}_article_rating" for m in MODELS]] == 0).sum(axis=1).value_counts().sort_index()

print("7.6% of articles are not nuclear-related" if round(unrelated_share * 100, 1) == 7.6 else "contradicted",
      f"{(cleaned['nuclear_related'] == 0).sum():,} of {len(cleaned):,} articles = {unrelated_share:.2%}")

zero_votes.rename_axis("models rating 0").rename("articles").to_frame()

7.6% of articles are not nuclear-related 2,015 of 26,483 articles = 7.61%


,articles
models rating 0,
0,21764
1,2704
2,1880
3,135


## Agreement between the annotators

> The annotators disagree noticeably: **Pearson correlations of the article rating between models are 0.52–0.69**.

In [10]:
article_columns = [f"{m}_article_rating" for m in MODELS]
all_pairs = df[article_columns].corr()
related_by_all = df[(df[article_columns] > 0).all(axis=1)]
related_pairs = related_by_all[article_columns].corr()

def pair_range(matrix):
    values = [matrix.iloc[i, j] for i in range(3) for j in range(i + 1, 3)]
    return min(values), max(values)

low, high = pair_range(all_pairs)
low_rel, high_rel = pair_range(related_pairs)
exact = (df[article_columns].nunique(axis=1) == 1).mean()
print(f"all three models give the identical article rating for {exact:.1%} of articles")
all_pairs.round(3)

all three models give the identical article rating for 40.4% of articles


,gemini_article_rating,deepseek_article_rating,gpt_article_rating
gemini_article_rating,1.000,0.556,0.686
deepseek_article_rating,0.556,1.000,0.518
gpt_article_rating,0.686,0.518,1.000


## Coverage per outlet

> Coverage is uneven, from **432 (Fakt_pl) to 5,207 (sme.sk)** articles per outlet.

In [11]:
per_outlet = df["source"].value_counts()
smallest, largest = per_outlet.idxmin(), per_outlet.idxmax()
per_year = dates.dt.year.value_counts().sort_index()
print("articles per year:", per_year.to_dict())
per_outlet.rename("articles").to_frame()

articles per year: {1998: 246, 1999: 9, 2000: 88, 2001: 312, 2002: 355, 2003: 442, 2004: 351, 2005: 421, 2006: 1052, 2007: 1143, 2008: 816, 2009: 756, 2010: 965, 2011: 2543, 2012: 1229, 2013: 1127, 2014: 1115, 2015: 1069, 2016: 988, 2017: 1043, 2018: 931, 2019: 1191, 2020: 797, 2021: 1152, 2022: 2814, 2023: 2116, 2024: 1407, 2025: 5}


,articles
source,
sme.sk,5207
index.hu,3741
origo.hun,3136
aktualne.cz,2749
Onet_pl,2350
novinky.cz,2155
iROZHLAS,1891
cas.sk,1614
pravda.sk,1401


## Duplicates

> **258 titles are duplicates.**

In [12]:
duplicate_titles = len(df) - df["title"].nunique()
print(f"Duplicated titles: {duplicate_titles}")
repeated_titles = (df["title"].value_counts() > 1).sum()
exact_duplicates = df.duplicated(["title", "article body"]).sum()
repeated_urls = df["url"].duplicated().sum()
outlets_per_title = df[df["title"].duplicated(keep=False)].groupby("title")["source"].nunique().value_counts()

df[df["title"].duplicated(keep=False)].sort_values("title")[["id", "source", "date", "title"]].head(6)

Duplicated titles: 258


,id,source,date,title
9818,15513,sme.sk,26-04-2011,25. výroie ernobyskej katastrofy
8309,15537,sme.sk,26-04-2011,25. výroie ernobyskej katastrofy
6177,11817,pravda.sk,05-09-2024,924. de: Rusko stiahlo z Ukrajiny 60-tisíc vojakov. Bielorusi opä zostreovali áhidy zo svojho neba
1679,11801,pravda.sk,05-09-2024,924. de: Rusko stiahlo z Ukrajiny 60-tisíc vojakov. Bielorusi opä zostreovali áhidy zo svojho neba
2789,21858,origo.hun,02-11-2009,A francia atomreaktorok közel harmada áll
5800,26747,index.hu,02-11-2009,A francia atomreaktorok közel harmada áll


## Body length cap

> **Source bodies are cut at 32,767 characters.**


In [13]:
lengths = body.str.len()
print(f"{lengths.max() = }")
at_cap = (lengths == 32767).sum()
near_cap = ((lengths >= 32000) & (lengths < 32767)).sum()

df.loc[lengths == 32767, "source"].value_counts().rename("articles at 32,767").to_frame()

lengths.max() = 32767


,"articles at 32,767"
source,
cas.sk,19
pravda.sk,10
sme.sk,6
aktualne.cz,4
index.hu,2
origo.hun,2
iROZHLAS,2
